This analysis presents acoustic and perceptual data from a study comparing vocal exercises derived from animal imitations (owl, cat, cow) with exercises derived from the corresponding vowel imitations (/u/, /æ/, /ɑ/). Five participants each completed two sessions — one animal-based, one vowel-based — with acoustic measures extracted from mimics, exercises, and a target song ("Somewhere Over the Rainbow") recorded at baseline and after each exercise block.

**Acoustic measures:** f0, jitter, shimmer, HNR, intensity (calibrated), spectral moments (CoG, spectral SD, skewness, kurtosis), CPPS, H1-H2, F1, F2, alpha ratio, L/H ratio, LTAS slope/tilt, AVQI.

**Perceptual measure:** Omni-VES (Vocal Effort Scale), rated by researcher and participant self-assessment.

**Statistical approach:** Cohen's d with bootstrapped 95% CIs, Spearman rank correlations. Emphasis on effect sizes and patterns over p-values given n = 5.

**Formant caveat:** Formant estimation is unreliable when f0 exceeds ~350 Hz. Owl mimics (~570 Hz) and "where" targets (~520 Hz) are flagged; cat mimics (~430 Hz) are borderline; cow mimics (~320 Hz) and "some" targets (~260 Hz) are reliable.

In [1]:
#| echo: false
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load data
df = pd.read_csv('data/combined_measures.csv')
r4a = pd.read_csv('data/results_4a_mimic_effects.csv')
r4b = pd.read_csv('data/results_4b_transfer_effects.csv')
r4e = pd.read_csv('data/results_4e_ves_correlations.csv')
r_extype = pd.read_csv('data/results_somewhere_by_exercise_type.csv')

# Display labels for measures
measure_labels = {
    'f0_mean': 'f0 Mean (Hz)',
    'f0_sd': 'f0 SD (Hz)',
    'jitter_local': 'Jitter (local)',
    'shimmer_local': 'Shimmer (local)',
    'shimmer_dB': 'Shimmer (dB)',
    'hnr_mean': 'HNR (dB)',
    'intensity_mean_calibrated': 'Intensity (dB SPL)',
    'cog': 'CoG (Hz)',
    'spectral_sd': 'Spectral SD (Hz)',
    'skewness': 'Skewness',
    'kurtosis': 'Kurtosis',
    'cpps': 'CPPS (dB)',
    'h1_h2': 'H1-H2 (dB)',
    'f1': 'F1 (Hz)',
    'f2': 'F2 (Hz)',
    'alpha_ratio': 'Alpha Ratio',
    'lh_ratio': 'L/H Ratio',
    'ltas_slope': 'LTAS Slope (dB/oct)',
    'ltas_tilt': 'LTAS Tilt (dB/oct)'
}

# Measure display order
measure_order = [
    'f0_mean', 'f0_sd', 'jitter_local', 'shimmer_local', 'shimmer_dB',
    'hnr_mean', 'intensity_mean_calibrated', 'cog', 'spectral_sd',
    'skewness', 'kurtosis', 'cpps', 'h1_h2', 'f1', 'f2',
    'alpha_ratio', 'lh_ratio', 'ltas_slope', 'ltas_tilt'
]

# Pair metadata
pair_order = ['Owl / /u/', 'Cat / /\u00e6/', 'Cow / /\u0251/']
pair_colors = {
    'Owl / /u/': '#3498db',
    'Cat / /\u00e6/': '#e67e22',
    'Cow / /\u0251/': '#27ae60'
}

# Participant colors
participant_colors = {
    'P1': '#e74c3c',
    'P2': '#3498db',
    'P3': '#27ae60',
    'P4': '#9b59b6',
    'P5': '#e67e22'
}

# Animal-vowel mapping
animal_vowel_map = {
    'owl': 'u',
    'cat': 'ae',
    'cow': 'a'
}

## Study Design

Five participants each completed two recording sessions:

- **Session A (Animal):** Imitate an animal sound (owl, cat, cow) → sing a derived exercise → sing "Somewhere Over the Rainbow"
- **Session B (Vowel):** Imitate the corresponding vowel (/u/, /æ/, /ɑ/) → sing a derived exercise → sing "Somewhere Over the Rainbow"

Three animal-vowel pairs target different vowels and vocal behaviors:

| Pair | Animal | Vowel | Approximate f0 | Primary Domain |
|------|--------|-------|----------------|----------------|
| Owl / /u/ | Owl hoo | /u/ | ~570 Hz | Intensity, stability |
| Cat / /æ/ | Meow | /æ/ | ~430 Hz | Resonance, spectrum |
| Cow / /ɑ/ | Moo | /ɑ/ | ~320 Hz | Phonation, registration |

Each session also included AVQI recordings (sustained vowel + continuous speech) before and after the exercise protocol, and "Somewhere Over the Rainbow" at baseline and after each of the three exercise blocks.

## Effect of Condition on Acoustic Measures (All Pairs Pooled)

Before breaking the data out by pair, this figure shows the simplest version of the story: what happens when we treat all animal mimics as one group and all vowel mimics as another? The horizontal bars show Cohen's d for animal-minus-vowel, with mimic phase and exercise phase side by side.

In [2]:
#| echo: false
# Figure 1 (Ian Howell): Collapsed effect sizes — animal vs. vowel, all pairs pooled
from scipy import stats as sp_stats

def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_sd = np.sqrt(((na - 1) * a.std(ddof=1)**2 + (nb - 1) * b.std(ddof=1)**2) / (na + nb - 2))
    if pooled_sd == 0:
        return 0.0
    return (a.mean() - b.mean()) / pooled_sd

collapsed_measures = [m for m in measure_order if m not in ('f1', 'f2')]  # exclude unreliable formants at high f0

results_collapsed = []
for phase in ['mimic', 'exercise']:
    phase_data = df[df['segment_type'] == phase]
    animal_data = phase_data[phase_data['condition'] == 'animal']
    vowel_data = phase_data[phase_data['condition'] == 'human']
    for meas in collapsed_measures:
        a = animal_data[meas].dropna()
        v = vowel_data[meas].dropna()
        if len(a) >= 3 and len(v) >= 3:
            d = cohens_d(a, v)
            results_collapsed.append({'phase': phase, 'measure': meas, 'd': d})

rc = pd.DataFrame(results_collapsed)

fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Mimic Phase', 'Exercise Phase'],
    shared_yaxes=True,
    horizontal_spacing=0.08
)

for col_idx, phase in enumerate(['mimic', 'exercise'], 1):
    phase_rc = rc[rc['phase'] == phase]
    # Order by measure_order
    phase_rc = phase_rc.set_index('measure').loc[[m for m in collapsed_measures if m in phase_rc['measure'].values]].reset_index()
    
    colors = ['#2196F3' if d > 0 else '#F44336' for d in phase_rc['d']]
    
    fig1.add_trace(go.Bar(
        y=[measure_labels.get(m, m) for m in phase_rc['measure']],
        x=phase_rc['d'],
        orientation='h',
        marker_color=colors,
        hovertemplate='%{y}<br>d = %{x:+.2f}<extra></extra>',
        showlegend=False
    ), row=1, col=col_idx)
    
    fig1.add_vline(x=0, line_dash='solid', line_color='black', line_width=1, row=1, col=col_idx)

fig1.update_yaxes(autorange='reversed', row=1, col=1)
fig1.update_yaxes(autorange='reversed', row=1, col=2)
fig1.update_xaxes(title_text="Cohen's d", row=1, col=1)
fig1.update_xaxes(title_text="Cohen's d", row=1, col=2)

fig1.update_layout(
    title=dict(text='Effect of Condition on Acoustic Measures (All Pairs Pooled)', font_size=16),
    template='plotly_white',
    height=550,
    margin=dict(l=180, t=80, b=60)
)

fig1.show()

Blue bars = animal mimic scored higher; red bars = vowel mimic scored higher. When all three pairs are pooled, animal mimics show less perturbation (lower jitter, lower shimmer), higher HNR, and more spectral spread in both phases. The exercise phase amplifies condition differences rather than washing them out — shimmer dB shows the largest exercise-phase effect.

This pooled view tells a clean story, but it hides important pair-specific differences. The next section disaggregates by pair to show that each animal mimic works through a different acoustic mechanism.

*Source: Ian Howell, Analysis Advisor*

## VES Ratings by Exercise Type

The Omni-VES (Voice Evaluation Scale) was rated by the researcher for every mimic and exercise segment. Participant self-ratings provide a second perspective. The box plots below show the distribution of VES ratings across all six exercise types (three animal, three vowel).

In [3]:
#| echo: false
# Figure 2 (Ian Howell): VES ratings by exercise type
ves_data = df[df['segment_type'].isin(['mimic', 'exercise'])].copy()
ves_data = ves_data.dropna(subset=['ves_researcher'])

# Order: animal types first, then vowel types
exercise_order = ['owl', 'cat', 'cow', 'u', 'ae', 'a']
exercise_display = {
    'owl': 'Owl', 'cat': 'Cat', 'cow': 'Cow',
    'u': '/u/', 'ae': '/æ/', 'a': '/ɑ/'
}
exercise_colors = {
    'owl': '#3498db', 'cat': '#e67e22', 'cow': '#27ae60',
    'u': '#85c1e9', 'ae': '#f0b27a', 'a': '#82e0aa'
}

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Researcher VES', 'Participant Self-Rating'],
    horizontal_spacing=0.1
)

for col_idx, (ves_col, label) in enumerate([('ves_researcher', 'Researcher'), ('ves_participant', 'Participant')], 1):
    ves_col_data = ves_data.dropna(subset=[ves_col])
    for ex_type in exercise_order:
        ex_vals = ves_col_data[ves_col_data['exercise_type'] == ex_type][ves_col]
        if len(ex_vals) > 0:
            fig2.add_trace(go.Box(
                y=ex_vals,
                name=exercise_display[ex_type],
                marker_color=exercise_colors[ex_type],
                boxmean=True,
                showlegend=False,
                hovertemplate=f'{exercise_display[ex_type]}<br>{label} VES: %{{y}}<extra></extra>'
            ), row=1, col=col_idx)

fig2.update_yaxes(title_text='VES Rating', dtick=1, range=[0.5, 7.5], row=1, col=1)
fig2.update_yaxes(dtick=1, range=[0.5, 7.5], row=1, col=2)

fig2.update_layout(
    title=dict(text='VES Ratings by Exercise Type and Condition', font_size=16),
    template='plotly_white',
    height=450,
    margin=dict(t=80, b=60)
)

fig2.show()

The VES gradient tracks vocal effort as expected: cow/ɑ (the most demanding pair — registration traversal, vocal fry, high pitch variability) consistently receives the highest effort ratings, while owl/u (the easiest — stable, projected, spectrally narrow) receives the lowest. Researcher ratings are higher and more gradient than self-ratings.

This confirms the VES is doing its job — capturing the degree of vocal effort required by each task. The more interesting question is which *acoustic* dimensions map onto perceived effort, and whether that mapping is consistent across vowel contexts (see VES–Acoustic Correlations below).

*Source: Ian Howell, Analysis Advisor*

## Each Animal Mimic Invites a Different Vocal Adjustment

The strongest finding in this study: the three animal mimics do not all work the same way. Each one changes the voice on different acoustic dimensions compared to its vowel counterpart.

The heatmap below shows Cohen's d effect sizes for animal vs. vowel mimics, broken out by pair. Blue = animal mimic is higher; red = vowel mimic is higher. Hover for confidence intervals.

In [4]:
#| echo: false
# Figure 4a: Mimic effect sizes by pair — interactive heatmap
pivot_d = r4a.pivot(index='measure', columns='pair', values='d')
pivot_ci_low = r4a.pivot(index='measure', columns='pair', values='ci_low')
pivot_ci_high = r4a.pivot(index='measure', columns='pair', values='ci_high')

# Reorder
measures_in_data = [m for m in measure_order if m in pivot_d.index]
pairs_in_data = [p for p in pair_order if p in pivot_d.columns]
pivot_d = pivot_d.loc[measures_in_data, pairs_in_data]
pivot_ci_low = pivot_ci_low.loc[measures_in_data, pairs_in_data]
pivot_ci_high = pivot_ci_high.loc[measures_in_data, pairs_in_data]

# Build hover text
hover_text = []
for m in measures_in_data:
    row = []
    for p in pairs_in_data:
        d_val = pivot_d.loc[m, p]
        ci_lo = pivot_ci_low.loc[m, p]
        ci_hi = pivot_ci_high.loc[m, p]
        row.append(
            f"{measure_labels.get(m, m)}<br>"
            f"d = {d_val:+.2f}<br>"
            f"95% CI: [{ci_lo:+.2f}, {ci_hi:+.2f}]"
        )
    hover_text.append(row)

fig_4a = go.Figure(data=go.Heatmap(
    z=pivot_d.values,
    x=pairs_in_data,
    y=[measure_labels.get(m, m) for m in measures_in_data],
    hovertext=hover_text,
    hoverinfo='text',
    colorscale='RdBu',
    zmid=0,
    zmin=-4,
    zmax=4,
    colorbar=dict(title=dict(text="Cohen's d", side='right'))
))

fig_4a.update_layout(
    title=dict(text='Mimic Phase Effect Sizes: Animal vs. Vowel by Pair', font_size=16),
    xaxis_title='Animal / Vowel Pair',
    yaxis=dict(autorange='reversed'),
    template='plotly_white',
    height=650,
    margin=dict(l=180, t=80, b=60)
)

fig_4a.show()

**Reading the heatmap:** Each cell shows the standardized difference (Cohen's d) between the animal mimic and its vowel counterpart. Positive values (blue) mean the animal mimic scored higher; negative values (red) mean the vowel mimic scored higher.

Three distinct acoustic signatures emerge:

- **Cow** changes *phonation*: HNR d = +3.91, f0 SD d = −2.55, lower perturbation across the board. The cow sound invites a fundamentally different way of using the voice compared to singing /ɑ/ — more modal, more stable, lower pitch.
- **Cat** changes *resonance and spectrum*: spectral SD d = +3.21, F2 d = +2.78, CoG d = +1.93. The meow concentrates energy in the upper spectrum in a way that singing /æ/ does not.
- **Owl** changes *intensity and stability*: intensity d = +2.57, shimmer dB d = −2.13. The owl hoo is essentially a more projected, more stable version of the vowel.

This is the most robust finding in the study. The effect sizes are very large (d > 2 for multiple measures across all three pairs) and consistent within pairs.

## Transfer from Mimic to Exercise

Do the acoustic differences established during mimicry carry over into the derived exercises? The side-by-side heatmaps below compare effect sizes during the mimic phase (left) and exercise phase (right) for each pair.

In [5]:
#| echo: false
# Figure 4b: Transfer heatmap — mimic vs exercise phase
r4b_mimic = r4b[r4b['phase'] == 'mimic'].copy()
r4b_exercise = r4b[r4b['phase'] == 'exercise'].copy()

# Pivot both
def pivot_phase(phase_df):
    piv = phase_df.pivot(index='measure', columns='pair', values='d')
    ms = [m for m in measure_order if m in piv.index]
    ps = [p for p in pair_order if p in piv.columns]
    return piv.loc[ms, ps], ms, ps

piv_mimic, ms_m, ps_m = pivot_phase(r4b_mimic)
piv_exercise, ms_e, ps_e = pivot_phase(r4b_exercise)

# Use common measures
common_measures = [m for m in measure_order if m in piv_mimic.index and m in piv_exercise.index]
common_pairs = [p for p in pair_order if p in piv_mimic.columns and p in piv_exercise.columns]

fig_4b = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Mimic Phase', 'Exercise Phase'],
    horizontal_spacing=0.12
)

y_labels = [measure_labels.get(m, m) for m in common_measures]

# Mimic heatmap
fig_4b.add_trace(go.Heatmap(
    z=piv_mimic.loc[common_measures, common_pairs].values,
    x=common_pairs,
    y=y_labels,
    colorscale='RdBu',
    zmid=0, zmin=-4, zmax=4,
    colorbar=dict(title='d', len=0.5, x=0.44),
    hovertemplate='%{y}<br>%{x}<br>d = %{z:+.2f}<extra>Mimic</extra>'
), row=1, col=1)

# Exercise heatmap
fig_4b.add_trace(go.Heatmap(
    z=piv_exercise.loc[common_measures, common_pairs].values,
    x=common_pairs,
    y=y_labels,
    colorscale='RdBu',
    zmid=0, zmin=-4, zmax=4,
    colorbar=dict(title='d', len=0.5, x=1.02),
    hovertemplate='%{y}<br>%{x}<br>d = %{z:+.2f}<extra>Exercise</extra>'
), row=1, col=2)

fig_4b.update_yaxes(autorange='reversed', row=1, col=1)
fig_4b.update_yaxes(autorange='reversed', showticklabels=False, row=1, col=2)

fig_4b.update_layout(
    title=dict(text='Transfer: Mimic Phase vs. Exercise Phase Effect Sizes', font_size=16),
    template='plotly_white',
    height=650,
    margin=dict(l=180, t=80, b=60)
)

fig_4b.show()

Most mimic-phase effects attenuate when participants move to the exercise — the exercise heatmap is "cooler" (closer to zero) than the mimic heatmap. But the degree of transfer varies by pair:

- **Cat pair** preserves spectral features best: spectral SD and CoG maintain 65–82% of their mimic-phase effect size during exercises.
- **Cow pair** preserves voice quality partially: HNR retains ~24% of its mimic effect; shimmer dB retains ~43%.
- **Owl pair** shows the least transfer: intensity drops 84%; shimmer dB reverses entirely.

Some effects *emerge* in the exercise phase that weren't present in mimics — notably, Cow H1-H2 becomes meaningful only during exercises, suggesting the exercise develops a phonation-type difference that the mimic merely hints at.

## Mimic-to-Exercise Change (Individual Participants)

The heatmap above summarizes transfer as group-level effect sizes. The spaghetti plots below show what happens at the individual level: each line connects a participant's mean value during the mimic phase to their mean value during the exercise phase, separately for the animal condition (solid blue) and vowel condition (dashed orange). Group means are overlaid in bold.

In [6]:
#| echo: false
# Figure 4 (Ian Howell): Mimic-to-exercise individual spaghetti lines
spaghetti_measures = ['shimmer_dB', 'jitter_local', 'hnr_mean', 'intensity_mean_calibrated', 'spectral_sd', 'h1_h2']

mimic_ex = df[df['segment_type'].isin(['mimic', 'exercise'])].copy()

fig4_spag = make_subplots(
    rows=2, cols=3,
    subplot_titles=[measure_labels.get(m, m) for m in spaghetti_measures],
    vertical_spacing=0.18,
    horizontal_spacing=0.08
)

for idx, meas in enumerate(spaghetti_measures):
    row = idx // 3 + 1
    col = idx % 3 + 1
    
    for pid in ['P1', 'P2', 'P3', 'P4', 'P5']:
        for cond, color, dash, cond_label in [
            ('animal', '#2196F3', 'solid', 'Animal'),
            ('human', '#FF9800', 'dash', 'Vowel')
        ]:
            p_cond = mimic_ex[
                (mimic_ex['participant'] == pid) &
                (mimic_ex['condition'] == cond)
            ]
            mimic_mean = p_cond[p_cond['segment_type'] == 'mimic'][meas].mean()
            exercise_mean = p_cond[p_cond['segment_type'] == 'exercise'][meas].mean()
            
            if not np.isnan(mimic_mean) and not np.isnan(exercise_mean):
                fig4_spag.add_trace(go.Scatter(
                    x=['Mimic', 'Exercise'],
                    y=[mimic_mean, exercise_mean],
                    mode='lines+markers',
                    line=dict(color=color, width=1.5, dash=dash),
                    marker=dict(size=5),
                    opacity=0.4,
                    showlegend=False,
                    hovertemplate=f'{pid} ({cond_label})<br>Mimic: {mimic_mean:.4f}<br>Exercise: {exercise_mean:.4f}<extra></extra>'
                ), row=row, col=col)
    
    # Group means
    for cond, color, dash, name in [
        ('animal', '#1565C0', 'solid', 'Animal mean'),
        ('human', '#E65100', 'dash', 'Vowel mean')
    ]:
        cond_data = mimic_ex[mimic_ex['condition'] == cond]
        mimic_grp = cond_data[cond_data['segment_type'] == 'mimic'][meas].mean()
        exercise_grp = cond_data[cond_data['segment_type'] == 'exercise'][meas].mean()
        
        show_legend = idx == 0
        fig4_spag.add_trace(go.Scatter(
            x=['Mimic', 'Exercise'],
            y=[mimic_grp, exercise_grp],
            mode='lines+markers',
            line=dict(color=color, width=3.5, dash=dash),
            marker=dict(size=10),
            name=name,
            showlegend=show_legend,
            hovertemplate=f'{name}<br>Mimic: {mimic_grp:.4f}<br>Exercise: {exercise_grp:.4f}<extra></extra>'
        ), row=row, col=col)

fig4_spag.update_layout(
    title=dict(text='Mimic-to-Exercise Change (Individual Participants, All Pairs Pooled)', font_size=16),
    template='plotly_white',
    height=600,
    legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='right', x=1),
    margin=dict(t=100, b=60)
)

fig4_spag.show()

Shimmer dB shows the clearest condition divergence: the animal condition smooths (shimmer decreases from mimic to exercise) while the vowel condition roughens (shimmer increases). Individual variability is substantial — not every participant follows the group trend — but the group-level separation is consistent with the transfer heatmap above.

Intensity drops from mimic to exercise in both conditions, more steeply for animal (consistent with the owl pair's intensity effect washing out during exercises). H1-H2 shows the animal condition shifting toward breathier phonation during exercises while the vowel condition moves toward more pressed — foreshadowing the Cow H1-H2 divergence visible in the rep-by-rep trajectories below.

*Source: Ian Howell, Analysis Advisor*

## Rep-by-Rep Trajectories

Exercises are not static across repetitions. The plots below track five key measures across 3–6 reps for each pair, showing individual participant lines (thin) and group means (thick). The animal-derived exercise is shown in solid lines; the vowel-derived exercise in dashed lines.

In [7]:
#| echo: false
# Figure 4c: Rep-by-rep trajectories for key measures
trajectory_measures = ['f0_mean', 'jitter_local', 'h1_h2', 'cpps', 'alpha_ratio']
exercises = df[df['segment_type'] == 'exercise'].copy()

animal_types = ['owl', 'cat', 'cow']
vowel_types = ['u', 'ae', 'a']
pair_labels_short = ['Owl / /u/', 'Cat / /\u00e6/', 'Cow / /\u0251/']

for meas in trajectory_measures:
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=pair_labels_short,
        horizontal_spacing=0.08
    )
    
    for col_idx, (animal, vowel) in enumerate(zip(animal_types, vowel_types), 1):
        animal_ex = exercises[exercises['exercise_type'] == animal].dropna(subset=[meas])
        vowel_ex = exercises[exercises['exercise_type'] == vowel].dropna(subset=[meas])
        
        # Individual participant traces
        for pid in ['P1', 'P2', 'P3', 'P4', 'P5']:
            # Animal condition
            p_animal = animal_ex[animal_ex['participant'] == pid].sort_values('rep_number')
            if len(p_animal) > 0:
                fig.add_trace(go.Scatter(
                    x=p_animal['rep_number'],
                    y=p_animal[meas],
                    mode='lines',
                    line=dict(color=participant_colors[pid], width=1),
                    opacity=0.3,
                    showlegend=False,
                    hovertemplate=f'{pid} (animal)<br>Rep %{{x}}<br>{measure_labels.get(meas, meas)}: %{{y:.4f}}<extra></extra>'
                ), row=1, col=col_idx)
            
            # Vowel condition
            p_vowel = vowel_ex[vowel_ex['participant'] == pid].sort_values('rep_number')
            if len(p_vowel) > 0:
                fig.add_trace(go.Scatter(
                    x=p_vowel['rep_number'],
                    y=p_vowel[meas],
                    mode='lines',
                    line=dict(color=participant_colors[pid], width=1, dash='dash'),
                    opacity=0.3,
                    showlegend=False,
                    hovertemplate=f'{pid} (vowel)<br>Rep %{{x}}<br>{measure_labels.get(meas, meas)}: %{{y:.4f}}<extra></extra>'
                ), row=1, col=col_idx)
        
        # Group means
        animal_mean = animal_ex.groupby('rep_number')[meas].mean().reset_index()
        vowel_mean = vowel_ex.groupby('rep_number')[meas].mean().reset_index()
        
        show_legend = col_idx == 1
        
        fig.add_trace(go.Scatter(
            x=animal_mean['rep_number'],
            y=animal_mean[meas],
            mode='lines+markers',
            line=dict(color='#2c3e50', width=3),
            marker=dict(size=8),
            name='Animal mean',
            showlegend=show_legend,
            hovertemplate=f'Animal mean<br>Rep %{{x}}<br>{measure_labels.get(meas, meas)}: %{{y:.4f}}<extra></extra>'
        ), row=1, col=col_idx)
        
        fig.add_trace(go.Scatter(
            x=vowel_mean['rep_number'],
            y=vowel_mean[meas],
            mode='lines+markers',
            line=dict(color='#95a5a6', width=3, dash='dash'),
            marker=dict(size=8, symbol='diamond'),
            name='Vowel mean',
            showlegend=show_legend,
            hovertemplate=f'Vowel mean<br>Rep %{{x}}<br>{measure_labels.get(meas, meas)}: %{{y:.4f}}<extra></extra>'
        ), row=1, col=col_idx)
    
    fig.update_layout(
        title=dict(text=f'Exercise Trajectories: {measure_labels.get(meas, meas)}', font_size=16),
        template='plotly_white',
        height=400,
        legend=dict(orientation='h', yanchor='bottom', y=1.08, xanchor='right', x=1),
        margin=dict(t=100, b=60)
    )
    
    for i in range(1, 4):
        fig.update_xaxes(title_text='Rep', dtick=1, row=1, col=i)
    fig.update_yaxes(title_text=measure_labels.get(meas, meas), row=1, col=1)
    
    fig.show()

The most striking trajectory is **Cow H1-H2**: the cow-derived exercise becomes breathier across repetitions (H1-H2 increases) while the /ɑ/-derived exercise becomes more pressed (H1-H2 decreases). This divergence suggests that the motor learning happening across repetitions is itself condition-dependent — the animal mimic seeds a vocal pattern that evolves as the singer repeats the exercise.

Other observations:
- **Cat H1-H2** shows parallel downward trends — both conditions become more pressed, but the cat-derived exercise starts from a higher baseline.
- **Alpha ratio** diverges for Cow but not for Cat or Owl, consistent with Cow's phonation-domain effect.
- **Jitter** is generally stable across reps, with the animal condition lower (less perturbed) for Cow and Owl.

## Carry-Over into "Somewhere Over the Rainbow"

After each exercise block, participants sang "Somewhere Over the Rainbow." The plots below track acoustic measures on two sustained pitch targets — "some" (~260 Hz, reliable formants) and "where" (~520 Hz, unreliable formants) — from baseline through three post-exercise measurements.

Individual participant lines are shown (thin, colored) with the group mean overlaid (thick black = animal, thick gray dashed = vowel).

In [8]:
#| echo: false
# Figure 4d: Somewhere carry-over trajectories
somewhere_measures = ['jitter_local', 'shimmer_local', 'hnr_mean', 'cpps', 'h1_h2', 'alpha_ratio']

sw = df[df['segment_type'].str.startswith('somewhere')].copy()
sw['time_point'] = sw['segment_type'].map({
    'somewhere_baseline': 0,
    'somewhere_post_ex1': 1,
    'somewhere_post_ex2': 2,
    'somewhere_post_ex3': 3
})
tp_labels = {0: 'Baseline', 1: 'Post-Ex 1', 2: 'Post-Ex 2', 3: 'Post-Ex 3'}

for pitch_target in ['some', 'where']:
    target_hz = '~260 Hz' if pitch_target == 'some' else '~520 Hz'
    sw_target = sw[sw['pitch_target'] == pitch_target]
    
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=[measure_labels.get(m, m) for m in somewhere_measures],
        vertical_spacing=0.15,
        horizontal_spacing=0.08
    )
    
    for idx, meas in enumerate(somewhere_measures):
        row = idx // 3 + 1
        col = idx % 3 + 1
        
        for pid in ['P1', 'P2', 'P3', 'P4', 'P5']:
            for cond, dash in [('animal', 'solid'), ('human', 'dash')]:
                pdata = sw_target[
                    (sw_target['participant'] == pid) &
                    (sw_target['condition'] == cond)
                ].dropna(subset=[meas]).sort_values('time_point')
                if len(pdata) > 0:
                    fig.add_trace(go.Scatter(
                        x=pdata['time_point'],
                        y=pdata[meas],
                        mode='lines',
                        line=dict(color=participant_colors[pid], width=1, dash=dash),
                        opacity=0.35,
                        showlegend=False,
                        hovertemplate=f'{pid} ({cond})<br>%{{y:.4f}}<extra></extra>'
                    ), row=row, col=col)
        
        # Group means
        for cond, color, dash, name in [
            ('animal', '#2c3e50', 'solid', 'Animal'),
            ('human', '#95a5a6', 'dash', 'Vowel')
        ]:
            cond_data = sw_target[sw_target['condition'] == cond].dropna(subset=[meas])
            grp = cond_data.groupby('time_point')[meas].mean().reset_index()
            show_legend = idx == 0
            fig.add_trace(go.Scatter(
                x=grp['time_point'],
                y=grp[meas],
                mode='lines+markers',
                line=dict(color=color, width=3, dash=dash),
                marker=dict(size=7),
                name=f'{name} mean',
                showlegend=show_legend,
                hovertemplate=f'{name} mean<br>%{{y:.4f}}<extra></extra>'
            ), row=row, col=col)
        
        fig.update_xaxes(
            tickvals=[0, 1, 2, 3],
            ticktext=['BL', 'Post 1', 'Post 2', 'Post 3'],
            row=row, col=col
        )
    
    fig.update_layout(
        title=dict(
            text=f'Carry-Over into "Somewhere" \u2014 "{pitch_target}" target ({target_hz})',
            font_size=16
        ),
        template='plotly_white',
        height=550,
        legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='right', x=1),
        margin=dict(t=100, b=60)
    )
    
    fig.show()

Individual variability overwhelms any group-level carry-over pattern. This is a sample-size limitation (n = 5), not evidence of no carry-over. The individual trajectories show that some participants respond more consistently than others, and the direction of change is not uniform across participants or measures.

Despite the noisy group-level data, the researcher consistently perceived progressive improvement within sessions — reduced strain, better registration connection, more balanced resonance — in both protocols. The qualitative narrative tells a clearer story than the acoustic measures at this sample size.

## Carry-Over by Preceding Exercise Type

The Somewhere probes above are labeled Post-Ex 1, Post-Ex 2, Post-Ex 3, but these numbers refer to the *exercise type* (1 = owl/u, 2 = cat/ae, 3 = cow/a), not the temporal order. The actual sequence of exercises was randomized per participant and session. This means we can ask: **does the preceding exercise type predict the direction and magnitude of change in the subsequent Somewhere probe?**

The figures below show mean change from baseline for each acoustic measure, grouped by which exercise was performed immediately before the Somewhere probe. Because the order was randomized, exercise-type effects can be partially separated from cumulative warm-up (dosage) effects.

In [9]:
#| echo: false
# Carry-over by preceding exercise type — grouped bar charts
extype_measures = ['jitter_local', 'shimmer_dB', 'hnr_mean', 'f0_sd', 'spectral_sd']
extype_labels = {
    'jitter_local': 'Jitter (local)',
    'shimmer_dB': 'Shimmer (dB)',
    'hnr_mean': 'HNR (dB)',
    'f0_sd': 'f0 SD (Hz)',
    'spectral_sd': 'Spectral SD (Hz)'
}

exercise_type_colors = {
    'owl/u': '#3498db',
    'cat/ae': '#e67e22',
    'cow/a': '#27ae60'
}

exercise_type_display = {
    'owl/u': 'Owl / /u/',
    'cat/ae': 'Cat / /\u00e6/',
    'cow/a': 'Cow / /\u0251/'
}

for cond, cond_title in [('animal', 'Animal Condition (Session A)'), ('human', 'Vowel Condition (Session B)')]:
    cond_data = r_extype[r_extype['condition'] == cond]

    # Build subplot titles for a 2x3 grid: top = "some", bottom = "where"
    titles = [f'"some" \u2014 {extype_labels[m]}' for m in extype_measures[:3]] + \
             [f'"where" \u2014 {extype_labels[m]}' for m in extype_measures[:3]]

    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=titles,
        vertical_spacing=0.2,
        horizontal_spacing=0.1
    )

    for target_idx, target in enumerate(['some', 'where']):
        target_data = cond_data[cond_data['pitch_target'] == target]
        row = target_idx + 1

        for col_idx, m in enumerate(extype_measures[:3]):
            col = col_idx + 1
            change_col = f'{m}_change'
            if change_col not in target_data.columns:
                continue

            for ex_type in ['owl/u', 'cat/ae', 'cow/a']:
                ex_vals = target_data[target_data['preceding_exercise'] == ex_type][change_col].dropna()
                mean_val = ex_vals.mean() if len(ex_vals) > 0 else 0
                sd_val = ex_vals.std() if len(ex_vals) > 1 else 0
                n = len(ex_vals)

                fig.add_trace(go.Bar(
                    x=[exercise_type_display[ex_type]],
                    y=[mean_val],
                    error_y=dict(type='data', array=[sd_val], visible=True),
                    marker_color=exercise_type_colors[ex_type],
                    name=exercise_type_display[ex_type],
                    showlegend=(row == 1 and col == 1),
                    legendgroup=ex_type,
                    hovertemplate=(
                        f'{exercise_type_display[ex_type]}<br>'
                        f'Mean change: %{{y:+.4f}}<br>'
                        f'SD: {sd_val:.4f}<br>'
                        f'n = {n}<extra></extra>'
                    )
                ), row=row, col=col)

            fig.update_yaxes(title_text='\u0394 from baseline', row=row, col=col)
            fig.add_hline(y=0, line_dash='solid', line_color='gray', line_width=0.5, row=row, col=col)

    fig.update_layout(
        title=dict(text=f'Carry-Over by Preceding Exercise Type: {cond_title}', font_size=16),
        template='plotly_white',
        height=600,
        barmode='group',
        legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='right', x=1),
        margin=dict(t=120, b=60, l=80)
    )

    fig.show()

### Animal Condition

In the animal condition, the cow/a exercises produce the most distinctive carry-over pattern on Tier 1 measures (those that survived every robustness test in the audit):

- **Jitter:** Cow produces a tight cluster of near-zero changes (mean change +0.0002), while owl and cat produce wider spread. The variability across exercise types (spread d = 0.89) is larger than typical within-group variability.
- **f0 SD:** Cow is associated with reduced pitch instability (mean change -2.35 Hz), while cat and owl show less consistent patterns (spread d = 1.26).
- **HNR:** Cow-preceded probes show the largest HNR increase (+1.45 dB on "some"), consistent with cow's phonation-domain acoustic signature.

### Vowel Condition

The vowel condition (Session B) shows smaller and less consistent differences across exercise types. No single exercise type dominates, and the error bars overlap substantially. This is consistent with the overall finding that the animal mimics produce more acoustically distinct effects than their vowel counterparts.

### Temporal Position vs. Exercise Type

Because the exercise order was randomized, we can test whether changes correlate with temporal position (1st, 2nd, or 3rd exercise block) rather than exercise type. No Spearman correlation between temporal position and acoustic change reaches statistical significance in any condition-target-measure combination. This means the patterns above reflect what the singer *just did* (motor priming from the specific exercise), not how long they have been exercising (cumulative warm-up).

This is a meaningful distinction for pedagogy: it suggests that each exercise primes specific articulatory and phonatory patterns, and the most recent priming is what gets captured in the subsequent singing. The carry-over is exercise-specific, not dose-dependent.

## What Predicts Perceived Vocal Effort?

The heatmap below shows Spearman correlations between Omni-VES ratings and each acoustic measure, broken out by vowel group. The "all" column pools all vowel contexts; the individual columns show within-vowel relationships.

In [10]:
#| echo: false
# Figure 4e: VES-Acoustic correlations heatmap
vowel_order = ['all', 'u', 'ae', 'a']
vowel_display = {'all': 'All', 'u': '/u/', 'ae': '/æ/', 'a': '/ɑ/'}

pivot_rho = r4e.pivot(index='measure', columns='vowel', values='rho')
pivot_p = r4e.pivot(index='measure', columns='vowel', values='p')

measures_4e = [m for m in measure_order if m in pivot_rho.index]
vowels_4e = [v for v in vowel_order if v in pivot_rho.columns]

pivot_rho = pivot_rho.loc[measures_4e, vowels_4e]
pivot_p = pivot_p.loc[measures_4e, vowels_4e]

# Hover text with p-values
hover_text_4e = []
for m in measures_4e:
    row = []
    for v in vowels_4e:
        rho = pivot_rho.loc[m, v]
        p = pivot_p.loc[m, v]
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        row.append(
            f"{measure_labels.get(m, m)}<br>"
            f"\u03c1 = {rho:+.3f} ({sig})<br>"
            f"p = {p:.4f}"
        )
    hover_text_4e.append(row)

fig_4e = go.Figure(data=go.Heatmap(
    z=pivot_rho.values,
    x=[vowel_display.get(v, v) for v in vowels_4e],
    y=[measure_labels.get(m, m) for m in measures_4e],
    hovertext=hover_text_4e,
    hoverinfo='text',
    colorscale='RdBu',
    zmid=0,
    zmin=-0.8,
    zmax=0.8,
    colorbar=dict(title=dict(text='Spearman \u03c1', side='right'))
))

fig_4e.update_layout(
    title=dict(text='VES\u2013Acoustic Correlations by Vowel Group', font_size=16),
    xaxis_title='Vowel Context',
    yaxis=dict(autorange='reversed'),
    template='plotly_white',
    height=650,
    margin=dict(l=180, t=80, b=60)
)

fig_4e.show()

The pooled correlations ("All" column) appear counterintuitive: more shimmer and jitter correlate *positively* with higher VES scores, while HNR correlates *negatively*. But this is a between-vowel confound — the cow pair, which produces higher perturbation and lower HNR, also requires the most vocal effort.

Within vowels, the story changes:

- **For /ɑ/:** Shimmer (ρ = +0.50) and HNR (ρ = −0.50) still correlate with VES, but CPPS and H1-H2 also appear. Breathier phonation and lower cepstral prominence associate with higher perceived effort.
- **For /æ/:** Skewness, kurtosis, and intensity are the strongest predictors. Less spectral complexity associates with higher perceived effort.
- **For /u/:** The pattern partially *reverses*. HNR is positively correlated with VES, opposite to the pooled direction. LTAS tilt is the strongest predictor.

**The key finding is not that VES tracks effort — it's designed to — but that the acoustic correlates of perceived effort are vowel-dependent.** What constitutes "effortful" singing differs by vowel context. There is no single acoustic profile that predicts perceived effort across all conditions. This has implications for how we evaluate vocal exercises — effort criteria should be vowel-appropriate, not one-size-fits-all.

The qualitative notes reinforce this: owl mimics receive consistently positive quality descriptions ("ease," "lofted space," "relaxed") and correspondingly low effort ratings (researcher VES = 2 for 4 of 5 participants). Cow mimics receive notes about strain and effortful production and correspondingly high effort ratings. The VES and the qualitative observations agree — they're both capturing effort, through different lenses.

| Pair | Typical Researcher VES | Quality Language |
|------|----------------------|------------------|
| Owl / /u/ | 2 | "none to be heard" [strain], "good mimicry," "release and ease" |
| Cat / /æ/ | 3–4 | "twang/ping" alongside "pressing," "constriction" |
| Cow / /ɑ/ | 4–6 | "vocal fry," "pressing," "registration breaks" |

## AVQI Pre/Post

The Acoustic Voice Quality Index (AVQI) was measured from sustained vowel + continuous speech recordings before and after each session. Neither protocol harms overall voice quality.

In [11]:
#| echo: false
# AVQI slopegraph — individual participants + group means
avqi_data = df[df['segment_type'].isin(['avqi_pre', 'avqi_post'])].copy()
avqi_data = avqi_data.dropna(subset=['avqi_v0202'])
avqi_data['time'] = avqi_data['segment_type'].map({'avqi_pre': 0, 'avqi_post': 1})

fig_avqi = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Session A (Animal)', 'Session B (Vowel)'],
    horizontal_spacing=0.12
)

for col_idx, (cond, cond_label) in enumerate([('animal', 'Animal'), ('human', 'Vowel')], 1):
    cond_data = avqi_data[avqi_data['condition'] == cond]
    
    # Individual participants
    for pid in ['P1', 'P2', 'P3', 'P4', 'P5']:
        pdata = cond_data[cond_data['participant'] == pid].sort_values('time')
        if len(pdata) == 2:
            pre_val = pdata[pdata['time'] == 0]['avqi_v0202'].values[0]
            post_val = pdata[pdata['time'] == 1]['avqi_v0202'].values[0]
            change = post_val - pre_val
            fig_avqi.add_trace(go.Scatter(
                x=['Pre', 'Post'],
                y=[pre_val, post_val],
                mode='lines+markers',
                line=dict(color=participant_colors[pid], width=2),
                marker=dict(size=8),
                name=pid,
                showlegend=col_idx == 1,
                hovertemplate=f'{pid}<br>Pre: {pre_val:.2f}<br>Post: {post_val:.2f}<br>\u0394: {change:+.2f}<extra></extra>'
            ), row=1, col=col_idx)
    
    # Group mean
    grp = cond_data.groupby('time')['avqi_v0202'].mean().reset_index()
    if len(grp) == 2:
        pre_mean = grp[grp['time'] == 0]['avqi_v0202'].values[0]
        post_mean = grp[grp['time'] == 1]['avqi_v0202'].values[0]
        fig_avqi.add_trace(go.Scatter(
            x=['Pre', 'Post'],
            y=[pre_mean, post_mean],
            mode='lines+markers',
            line=dict(color='black', width=3),
            marker=dict(size=12, symbol='diamond'),
            name='Group mean',
            showlegend=col_idx == 1,
            hovertemplate=f'Mean<br>Pre: {pre_mean:.2f}<br>Post: {post_mean:.2f}<br>\u0394: {post_mean - pre_mean:+.2f}<extra></extra>'
        ), row=1, col=col_idx)

fig_avqi.update_yaxes(title_text='AVQI (v02.02)', row=1, col=1)
fig_avqi.update_yaxes(range=[1.5, 4.5], row=1, col=1)
fig_avqi.update_yaxes(range=[1.5, 4.5], row=1, col=2)

fig_avqi.update_layout(
    title=dict(text='AVQI Pre/Post by Condition', font_size=16),
    template='plotly_white',
    height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='right', x=1),
    margin=dict(t=100, b=60)
)

fig_avqi.show()

AVQI values are stable or slightly improved pre-to-post in both conditions. No participant shows a clinically meaningful worsening. This means neither the animal nor the vowel exercise protocol introduces vocal strain or degradation at the whole-voice level — an important baseline finding even if AVQI is too blunt an instrument to capture the fine-grained acoustic shifts documented above.

Note: P4 Session B (Vowel) post-AVQI was not available.

## Qualitative Observations

The researcher (Morgan) recorded observations on strain/tension, breath support, resonance/placement, and general impressions for every segment across all 10 sessions. These observations ground the acoustic findings in the language voice teachers actually use.

### Themes by Animal-Vowel Pair

**Owl / /u/ — "Ease," "lofted space," "relaxed"**

The owl mimic consistently elicited the most positive quality language. Descriptions include: "none to be heard" (re: strain), "a good amount of release and ease," "good lofted, back mouth space," and "all four reps are very similar in timbre, pitch, breath control." The owl exercises continued this pattern: "still sounded easeful," "relatively free, easy, relaxed vibrato and breath." Acoustically, this maps to low perturbation, high intensity, and spectrally narrow output.

**Cat / /æ/ — "Twang/ping," "pressing," "constriction"**

The cat mimic produced mixed observations — positive resonance features alongside effortful production: "some good ping starting to develop" but also "some tightness potentially around the larynx/pharynx" and "some pharyngeal constriction." Multiple participants naturally produced an /m/ onset, reflecting the natural onset of "meow." Acoustically, this maps to high spectral SD, high CoG, and high F2 — the meow engages M2 resonance but sometimes through effort rather than ease.

**Cow / /ɑ/ — "Vocal fry," "pressing in fry," "registration breaks"**

The cow mimic is defined by registration traversal — chest voice through fry into head voice. Notes consistently flag: "some strain in the fry/chest," "brightness in the chest/warmth in the head — not much connection throughout," and "fry to M2 to fry." The exercises show the pattern continuing: "seems to be some pressing at the vocal fold level here." This maps to the HNR/f0 SD/perturbation pattern, and the H1-H2 divergence across reps.

### "Somewhere" Trajectory — What the Listener Heard

Despite noisy group-level acoustic data, the researcher consistently perceived progressive improvement:

- P1: Baseline "some adduction issues" → Post-Cow "less overall tension, better phrasing, more focus and ping, seems to be an improvement in overall function"
- P2: Baseline "some slight strain on ascending leaps" → Post-Cow "much more ease in transition areas, the connection between the larger leaps were very successful"
- P4: Baseline "slight strain on ascending leaps" → Post-Cow "the least amount of strain yet! Registration is connected, best take yet!"
- P5: Baseline "excessive airy quality due to hypofunction" → Post-Cat "we can hear the mechanism making adjustments here in real time"

Both protocols appear to warm up the voice and build coordination. The *kind* of improvement may differ — the animal session developing more M2 access and registration connection, the vowel session developing more breath flow and release — but both trajectories point toward improved function.

## Summary

| Finding | Confidence | Evidence |
|---------|-----------|----------|
| The three animal-vowel pairs work through different acoustic mechanisms | **High** | Very large effect sizes (d > 2 for multiple measures across all three pairs), consistent within pairs |
| Animal mimics produce acoustically distinct vocalizations vs. vowel mimics | **High** | Every pair shows at least two measures with d > 1.5 |
| Exercises partially inherit the mimic's acoustic signature | **Moderate** | Transfer is real but attenuated and pair-dependent; Cat spectral features transfer best |
| Neither protocol harms overall voice quality | **High** | AVQI stable or improved pre-to-post in both conditions |
| The acoustic correlates of perceived effort are vowel-dependent | **Moderate** | Different acoustic dimensions predict VES for each vowel; no single "effort profile" across conditions |
| Cow H1-H2 diverges across exercise reps | **Exploratory** | Visible in trajectories but may be driven by 1-2 participants at n = 5 |
| Carry-over into "Somewhere" is exercise-type-specific, not dose-dependent | **Exploratory** | Cow/a exercises produce the most distinctive carry-over on Tier 1 measures (jitter spread d = 0.89, f0 SD spread d = 1.26); no temporal position correlation reaches significance |
| Carry-over into "Somewhere" is not clearly detectable at the group level | **Noted** | Individual variability overwhelms group signal; perceptual improvement perceived by researcher |

### Limitations

- **Sample size (n = 5):** Effect sizes are unstable, confidence intervals wide. Findings describe these five singers, not a generalizable population.
- **Order effects:** Cannot fully separate condition effects from session-order effects.
- **Formant reliability:** Unreliable at f0 > 350 Hz (owl mimics, "where" targets).
- **Single rater:** VES rated by one researcher without inter-rater reliability.

This study is designed as a proof-of-concept and pattern identification, not as a definitive test of the hypothesis. The acoustic signatures are robust; the generalizability awaits replication with a larger sample.